# Course 17 — End-to-End Agentic PDLC

Take Northstar change **AI-2410** from approved intent to an independently verified risk-summary proposal. The default path is deterministic, credential-free, and intentionally stops before merge or release authority.

![Controlled lifecycle from intent through integration review](assets/end-to-end-pdlc.svg)

## Objectives, safety boundary, and reproducibility

You will inspect readiness states, compile exact context, compare a direct-coding baseline, validate grounding and authority, inject seven failures, and trace specification-change invalidation. Model output proposes; trusted code rechecks evidence authorization, validates exact claim support, resolves approval through a trusted registry, consumes review authority, and records state. No code cell calls an external model or service.

In [ ]:
import importlib.util
import json
import sys
from dataclasses import asdict
from pathlib import Path

lab_spec = importlib.util.spec_from_file_location('course17_notebook_lab', Path('lab.py'))
lab = importlib.util.module_from_spec(lab_spec)
sys.modules[lab_spec.name] = lab
lab_spec.loader.exec_module(lab)
from app.generator import DeterministicRiskGenerator
from app.validator import validate_proposal

print('offline mode:', True)
print('change:', 'AI-2410')
print('evaluation time:', lab.EVALUATION_TIME)

## 1. Start with evidence, not implementation guesses

The fixture includes authorized evidence, another submission, a restricted source, and instruction-shaped document text. The context builder—not the model—selects what belongs to the subject.

In [ ]:
documents = lab.load_documents()
context, selected = lab.reference_context()
print('all fixture documents:', [item.document_id for item in documents])
print('authorized context:', sorted(selected))
print('context digest:', context.digest)
assert 'DOC-OTHER-001' not in selected
assert 'DOC-RESTRICTED-001' not in selected

The cross-submission and restricted documents never enter model-facing context. Authorization precedes semantic selection and is independently rechecked for every citation during validation. The fixture flag represents a trusted evidence service; production must not trust document- or caller-supplied authorization.

## 2. Inspect the minimal proposal contract

The generator can produce claims and citations. It has no application method for underwriting mutation or broker delivery. A typed object still needs independent validation; shape alone is not trust.

In [ ]:
proposal = DeterministicRiskGenerator().generate(context, selected)
print(json.dumps(asdict(proposal), indent=2))
assert proposal.attempted_actions == ()

## 3. Compare direct coding with governed delivery

The direct baseline treats structured output as completed work. The governed path checks the same seven-case population for exact context, grounding, and forbidden effects.

In [ ]:
comparison = lab.compare_baseline()
print('population:', comparison['population'])
print('governed accuracy:', comparison['governed_accuracy'])
print('baseline independently validated:', comparison['direct_baseline_validated'])
for row in comparison['cases']:
    print(row['case'], '→', row['governed_state'], row['governed_findings'])
assert comparison['governed_correct'] == comparison['population']

This result is a deterministic fixture comparison, not a model benchmark. The baseline's problem is not that every output is wrong; it is that it produces no independent basis for distinguishing safe from unsafe outputs.

## 4. Execute the reference path

A valid proposal passes independent validation. The caller supplies only an opaque receipt ID; a trusted registry supplies authenticated approval facts bound to the exact proposal. The strongest state is integration-review eligibility.

In [ ]:
reference, trace = lab.run_pipeline()
print(json.dumps(trace, indent=2))
assert reference.state == 'READY_FOR_INTEGRATION_REVIEW'
assert reference.release_authorized is False

## 5. Run the seven required experiments

Each experiment changes one meaningful property: citation existence, evidence support, injection-shaped content, context identity, protected effects, or governing specification revision.

In [ ]:
experiments = lab.run_experiments()
for name, result in experiments.items():
    print(name, '→', result['state'], result.get('validation_findings', []))
assert experiments['valid_summary']['state'] == 'READY_FOR_INTEGRATION_REVIEW'
assert experiments['missing_source']['state'] == 'BLOCKED'
assert experiments['unsupported_claim']['state'] == 'BLOCKED'
assert experiments['prompt_injection']['state'] == 'BLOCKED'
assert experiments['prompt_injection']['attempted_actions'] == ['send_broker_email']
assert experiments['prompt_injection']['effect_invocations'] == []
assert experiments['stale_context']['state'] == 'BLOCKED'
assert experiments['unauthorized_mutation']['state'] == 'BLOCKED'
assert experiments['specification_change']['revalidation_required']

The adversarial document reaches the generator and induces an attempted broker action. Trusted validation rejects the attempt, and the recording gateway proves that no external effect ran. Safety is established at the application boundary, not by assuming the generator ignores malicious text.

## 6. Inspect controlled convergence

Specification revision 4 produces a new context digest. The old proposal fails against both identities. Verification cannot make revision 3 authoritative again.

In [ ]:
changed, change_trace = lab.run_specification_change()
print(json.dumps(change_trace, indent=2))
assert changed.state == 'BLOCKED'
assert {'STALE_CONTEXT', 'STALE_SPECIFICATION'}.issubset(change_trace['validation_findings'])
assert change_trace['specification_authority_preserved']

## 7. Evaluate represented behavior

Golden scenarios test end-to-end states. Mutation cases test independent control families and near-boundary variants. The reported denominators describe these fixtures only.

In [ ]:
conformance = lab.run_conformance()
mutations = lab.run_mutation_evaluation()
print('golden:', conformance['passed'], '/', conformance['total'])
print('mutations:', mutations['passed'], '/', mutations['total'])
print('claim:', mutations['claim'])
assert (conformance['passed'], conformance['total']) == (12, 12)
assert (mutations['passed'], mutations['total']) == (39, 39)

## 8. Exercise delivery, replay, and semantic traceability

The lab creates three actual commits in a temporary Git repository and invalidates their integration evidence after a prerequisite revision. It also proves durable receipt replay protection across a store restart and detects four semantically wrong trace links.

In [ ]:
delivery = lab.run_three_commit_exercise()
restart = lab.run_restart_replay_exercise()
traceability = lab.run_traceability_evaluation()
print('dependency order:', delivery['dependency_order_valid'])
print('after prerequisite change:', delivery['invalidation_code'])
print('replay after restart:', restart['reason_codes_after_restart'])
print('semantic trace mutations:', traceability['passed'], '/', traceability['total'])
assert delivery['invalidation_code'] == 'STALE_CONTRACT_REVISION'
assert restart['atomic_uniqueness_enforced']
assert traceability['reference_findings'] == []
assert (traceability['passed'], traceability['total']) == (4, 4)

## 9. Production upgrade and exercises

Replace fixture identity with authenticated tenant and workload state; local run digests with signed attestations; structured exact matching with a calibrated domain support evaluator; the in-process registry and local SQLite exercise with managed identity and durable atomic storage; and the deterministic generator with an approved, versioned gateway plus representative evaluation. Add retries, cancellation, concurrency control, recovery, privacy-safe telemetry, build provenance, deployment authorization, rollout, and runtime evidence.

Exercises:
1. Add a contradictory structured fact and preserve contradicted versus insufficient outcomes.
2. Add a valid-work-blocked metric and four boundary cases.
3. Compare one-agent and two-agent schedules for the AWU graph.
4. Complete `workshop/starter/lifecycle-package.json` without authorizing release.
5. Map the same artifacts into Spec Kit and OpenSpec while preserving authority.

## Summary

Business approval starts discovery. Specification readiness defines behavior. Bounded work units and pinned context constrain execution. Independent evidence supports integration review. Merge and release remain separate trusted decisions.